# Building your CESDM Model (Proxy API Notebook)

Interactive companion to the **Building your CESDM Model** tutorial (Parts 1–4). Builds the Switzerland + neighbours reference model using the **Proxy API** — assign attributes and relations on typed entity handles returned by `model.add_entity(...)`.

**Prerequisites:** `pip install -e ".[jupyter]"` from the repository root. Start Jupyter from `cesdm-toolbox/` so paths to `schemas/cesdm/` resolve.

Core EAR script equivalent: `docs/examples/reference_energy_system_model_core_api.py`


## 0. Repository setup

The following cell locates the CESDM repository, adds it to the Python path, and
imports the current toolbox.

Run the notebook from inside the repository or place it under `docs/examples/`.

In [1]:
from cesdm_toolbox import build_model_from_yaml
from cesdm.default_library import GeneratorTypes, Carriers

## 1. Load the schema and Default Library

The schema defines the permitted classes, attributes, relations, inheritance, and
constraints. The Default Library adds reusable entity instances such as carriers,
natural resources, and technology definitions.

No project-specific physical assets exist yet.

In [2]:
model = build_model_from_yaml("schemas/cesdm")
model.import_library("library/default_library")

print("Schema and Default Library loaded.")
print(model.summary())

NameError: name 'SCHEMA_DIR' is not defined

## Inspecting the model while it grows

After each major modelling step, the notebook prints a compact view of the entities that were added.
The helper functions below do not modify the model; they only expose its current state.

In [ ]:
def iter_entity_ids(model):
    """Yield all entity identifiers grouped internally by entity class."""
    for entities_by_id in (model.entities or {}).values():
        for entity_id in (entities_by_id or {}):
            yield entity_id


def print_entity(model, entity_id: str) -> None:
    """Print one entity with schema-aware attributes and relations."""
    print("=" * 80)
    print(entity_id)
    print("=" * 80)

    entity_class = model.entity_class(entity_id)
    if entity_class is None:
        print("Entity not found.")
        return

    data = model.entity_data(entity_id)
    attribute_ids = set(model.class_attributes(entity_class))
    relation_ids = set(model.class_relations(entity_class))

    print(f"Class: {entity_class}")

    print("\nAttributes:")
    found_attribute = False
    for attribute_id in sorted(attribute_ids):
        if attribute_id not in data:
            continue

        found_attribute = True
        stored = data[attribute_id]

        if isinstance(stored, dict) and "value" in stored:
            value = stored.get("value")
            unit = stored.get("unit")
            provenance = stored.get("provenance_ref")
        else:
            value = stored
            unit = None
            provenance = None

        suffix = f" {unit}" if unit else ""
        print(f"  - {attribute_id}: {value}{suffix}")

        if provenance:
            print(f"      provenance: {provenance}")

    if not found_attribute:
        print("  (none)")

    print("\nRelations:")
    found_relation = False
    for relation_id in sorted(relation_ids):
        targets = model.get_relation_targets(entity_id, relation_id)
        if not targets:
            continue

        found_relation = True
        for target in targets:
            print(f"  - {relation_id} -> {target}")

    if not found_relation:
        print("  (none)")


def print_class_counts(model, title: str = "Current model") -> None:
    """Print entity counts grouped by schema class."""
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    counts = {
        class_name: len(entities_by_id or {})
        for class_name, entities_by_id in (model.entities or {}).items()
        if entities_by_id
    }

    for class_name in sorted(counts):
        print(f"{class_name:30s} {counts[class_name]:>4}")

    print("-" * 80)
    print(f"{'Total entities':30s} {sum(counts.values()):>4}")

In [ ]:
print_class_counts(model, "After loading schema and Default Library")

## 2. Create the system and electricity Carrier Domain

The first project-specific entities are the system container and the electricity
Carrier Domain.

This section illustrates the basic EAR pattern:

1. create an entity;
2. describe it with attributes;
3. connect it to another entity through a relation.

In [ ]:
system = model.add_entity("EnergySystemModel", "CH_NEIGHBOURS_2030")
system.long_name = "CH + neighbours multi-domain energy system, 2030"
system.co2_price = 80.0

electricity_domain = model.add_entity("CarrierDomain", "domain.electricity")
electricity_domain.name = "Electricity"
electricity_domain.hasCarrier = Carriers.CARRIER_ELECTRICITY

### Outcome
The model now has a project container and an electricity Carrier Domain. The domain references the reusable electricity Carrier from the Default Library.

In [ ]:
print_entity(model, "CH_NEIGHBOURS_2030")
print_entity(model, "domain.electricity")
print_class_counts(model, "After system and electricity domain")

## 3. Add geographical regions

Switzerland and its neighbouring countries are represented as uniquely identified
`GeographicalRegion` entities.

In [ ]:
countries = [
    ("region.ch", "Switzerland"),
    ("region.de", "Germany"),
    ("region.fr", "France"),
    ("region.it", "Italy"),
    ("region.at", "Austria"),
]
for region_id, name in countries:
    region = model.add_entity("GeographicalRegion", region_id)
    region.name = name

### Outcome
Five geographical entities were added. They can now be used as targets of spatial relations such as `locatedIn`.

In [ ]:
print_entity(model, "region.ch")
print_class_counts(model, "After regions")

## 4. Build the electricity network

Each country receives one aggregated high-voltage `ElectricalBus`. Attributes
describe the bus, while relations place it in a geographical region and the
electricity Carrier Domain.

In [ ]:
buses = [
    ("bus.ch", "region.ch", "Switzerland 380kV", 380.0, 47.0, 8.0),
    ("bus.de", "region.de", "Germany 380kV", 380.0, 51.0, 10.0),
    ("bus.fr", "region.fr", "France 400kV", 400.0, 46.0, 2.0),
    ("bus.it", "region.it", "Italy 380kV", 380.0, 42.0, 12.0),
    ("bus.at", "region.at", "Austria 380kV", 380.0, 47.5, 14.0),
]
for bus_id, region_id, name, voltage_kv, latitude, longitude in buses:
    bus = model.add_entity("ElectricalBus", bus_id)
    bus.name = name
    bus.nominal_voltage = (voltage_kv, "kV")
    bus.latitude = latitude
    bus.longitude = longitude
    bus.locatedIn = model.get_entity(region_id)
    bus.belongsToCarrierDomain = electricity_domain

### Outcome
One aggregated high-voltage bus was created for each country. The Swiss bus combines technical, spatial, and domain information without becoming separate objects.

In [ ]:
print_entity(model, "bus.ch")
print_class_counts(model, "After electricity buses")

## 5. Add electricity demand

Each `DemandUnit` is assigned an annual energy demand and connected to its
electrical bus through `atNode`.

In [ ]:
demands = [
    ("dem.ch", "CH electricity demand", 60_000, "bus.ch"),
    ("dem.de", "DE electricity demand", 500_000, "bus.de"),
    ("dem.fr", "FR electricity demand", 450_000, "bus.fr"),
    ("dem.it", "IT electricity demand", 300_000, "bus.it"),
    ("dem.at", "AT electricity demand", 70_000, "bus.at"),
]
for demand_id, name, annual_gwh, bus_id in demands:
    demand = model.add_entity("DemandUnit", demand_id)
    demand.name = name
    demand.annual_energy_demand = (annual_gwh * 1000, "MWh/year")
    demand.atNode = model.get_entity(bus_id)

### Outcome
Each country now has an aggregated `DemandUnit`. Annual demand describes its magnitude, while `atNode` locates it in the network.

In [ ]:
print_entity(model, "dem.ch")
print_class_counts(model, "After electricity demand")

## 6. Define the shared time axis

A single hourly `TimestampSeries` defines the temporal index for all 2030 profiles.
Wind, solar, and water resources are reused from the imported Default Library and
are not recreated here.

In [ ]:
timestamps = model.add_entity("TimestampSeries", "ts.hourly.2030")
timestamps.name = "Hourly, 2030"
timestamps.start_datetime = "2030-01-01T00:00:00"
timestamps.resolution = "PT1H"
timestamps.length = 8760
timestamps.timezone = "Europe/Zurich"

### Outcome
A shared hourly time axis for 2030 is available. Multiple Profiles can reference it, keeping temporal assumptions consistent.

In [ ]:
print_entity(model, "ts.hourly.2030")
print_class_counts(model, "After shared time axis")

## 7. Add the generation fleet and availability profiles

Physical generators contain asset-specific data such as name, capacity, and
location. Shared technology properties are referenced from the Default Library
through `hasTechnology`.

Renewable availability profiles are created explicitly as `Profile` entities and
linked to the common `TimestampSeries`.

In [ ]:
generators = [
    (
        "gen.ch.gas",
        "CH Gas CCGT",
        GeneratorTypes.GENERATION_THERMAL_GAS_CCGT_NEW,
        3_000,
        "bus.ch",
        "thermal",
        None,
    ),
    (
        "gen.ch.nuc",
        "CH Nuclear",
        GeneratorTypes.GENERATION_THERMAL_NUCLEAR_STANDARD,
        2_000,
        "bus.ch",
        "nuclear",
        None,
    ),
    (
        "gen.ch.wind",
        "CH Wind",
        GeneratorTypes.GENERATION_RENEWABLE_WIND_ONSHORE,
        500,
        "bus.ch",
        "wind",
        900_000,
    ),
    (
        "gen.ch.solar",
        "CH Solar PV",
        GeneratorTypes.GENERATION_RENEWABLE_SOLAR_PV_UTILITY,
        2_000,
        "bus.ch",
        "solar",
        2_000_000,
    ),
]

for (
    generator_id,
    name,
    technology_id,
    capacity_mw,
    bus_id,
    family,
    annual_mwh,
) in generators:
    generator = model.add_entity("GenerationUnit", generator_id)
    generator.name = name
    generator.nominal_power_capacity = (capacity_mw, "MW")
    generator.hasTechnology = technology_id
    generator.atNode = model.get_entity(bus_id)

    if family == "thermal":
        generator.hasInputCarrier = Carriers.CARRIER_FUEL_FOSSIL_GAS_NATURAL_GAS
    elif family == "wind":
        generator.hasInputResource = "resource.renewable.wind"
    elif family == "solar":
        generator.hasInputResource = "resource.renewable.solar"

    if annual_mwh is not None:
        generator.annual_resource_potential = annual_mwh

        profile_id = f"profile.{generator_id}.capacity_factor"
        profile = model.add_entity("Profile", profile_id)
        profile.profile_type = "as_capacity_factor"
        profile.profile_unit = "pu"
        profile.data_reference = f"/profiles/{profile_id}"
        profile.hasTimestampSeries = timestamps
        generator.hasAvailabilityProfile = profile

## 8. Represent reservoir hydro

The reservoir and hydro turbine are modelled as separate physical entities.
Explicit relations preserve the connection between storage, resource, generation,
and the natural-inflow profile.

In [ ]:
reservoir_id = "storage.ch.hydro.reservoir"
hydro_id = "gen.ch.hydro.reservoir"
inflow_profile_id = f"profile.{reservoir_id}.inflow"

reservoir = model.add_entity("ReservoirStorageUnit", reservoir_id)
reservoir.name = "CH Alpine seasonal reservoir"
reservoir.energy_storage_capacity = (8_800_000, "MWh")
reservoir.annual_natural_inflow_energy = (20_000_000, "MWh/year")
reservoir.storesResource = "resource.water"

hydro = model.add_entity("HydroGenerationUnit", hydro_id)
hydro.name = "CH Reservoir hydro turbines"
hydro.machine_role = "turbine"
hydro.nominal_power_capacity = (8_000, "MW")
hydro.annual_resource_potential = 20_000_000
hydro.hasTechnology = "Generation.Renewable.Hydro.Reservoir"
hydro.hasInputResource = "resource.water"
hydro.atNode = model.get_entity("bus.ch")
hydro.drawsFromReservoir = reservoir
reservoir.suppliesResourceTo = hydro

inflow_profile = model.add_entity("Profile", inflow_profile_id)
inflow_profile.profile_type = "as_normalized_annual_energy"
inflow_profile.profile_unit = "pu"
inflow_profile.data_reference = f"/profiles/{inflow_profile_id}"
inflow_profile.hasTimestampSeries = timestamps
reservoir.hasNaturalInflowProfile = inflow_profile

### Outcome
Reservoir hydro is represented by separate storage, turbine, and Profile entities. Explicit relations preserve their physical association.

In [ ]:
print_entity(model, "storage.ch.hydro.reservoir")
print_entity(model, "gen.ch.hydro.reservoir")
print_entity(model, "profile.storage.ch.hydro.reservoir.inflow")
print_class_counts(model, "After reservoir hydro")

## 9. Add cross-border interconnectors

Each `Interconnector` explicitly defines its two endpoint buses and directional
transfer capacities. No `connect()` convenience helper is used.

In [ ]:
interconnectors = [
    ("ntc.ch.de", "CH-DE NTC", "bus.ch", "bus.de", 6_000, 5_500),
    ("ntc.ch.fr", "CH-FR NTC", "bus.ch", "bus.fr", 4_000, 3_500),
    ("ntc.ch.it", "CH-IT NTC", "bus.ch", "bus.it", 5_000, 4_500),
    ("ntc.ch.at", "CH-AT NTC", "bus.ch", "bus.at", 2_000, 2_000),
]
for (
    interconnector_id,
    name,
    from_bus,
    to_bus,
    capacity_from_to,
    capacity_to_from,
) in interconnectors:
    interconnector = model.add_entity("Interconnector", interconnector_id)
    interconnector.name = name
    interconnector.maximum_power_flow_from_to = (capacity_from_to, "MW")
    interconnector.maximum_power_flow_to_from = (capacity_to_from, "MW")
    interconnector.fromNode = model.get_entity(from_bus)
    interconnector.toNode = model.get_entity(to_bus)

### Outcome
Switzerland is connected to its neighbours through explicit Interconnector entities with directional transfer capacities.

In [ ]:
print_entity(model, "ntc.ch.de")
print_class_counts(model, "After interconnectors")

## 10. Add gas and heat Carrier Domains

The system is extended beyond electricity by creating gas and heat domains and
their corresponding network nodes with the same three EAR operations.

In [ ]:
heat_carrier = model.add_entity("Carrier", "carrier.heat")
heat_carrier.name = "Heat"

domains = [
    (
        "domain.gas.ch",
        "Swiss gas domain",
        Carriers.CARRIER_FUEL_FOSSIL_GAS_NATURAL_GAS,
    ),
    (
        "domain.heat.ch",
        "Swiss heat domain",
        "carrier.heat",
    ),
]
domain_entities = {}
for domain_id, name, carrier_id in domains:
    domain = model.add_entity("CarrierDomain", domain_id)
    domain.name = name
    domain.hasCarrier = carrier_id
    domain_entities[domain_id] = domain

region_ch = model.get_entity("region.ch")

nodes = [
    ("GasBus", "bus.ch.gas", "Swiss gas bus", "domain.gas.ch"),
    ("HeatBus", "bus.ch.heat", "Swiss heat bus", "domain.heat.ch"),
]
for entity_class, node_id, name, domain_id in nodes:
    node = model.add_entity(entity_class, node_id)
    node.name = name
    node.locatedIn = region_ch
    node.belongsToCarrierDomain = domain_entities[domain_id]

### Outcome
The model now contains electricity, gas, and heat Carrier Domains. Each domain has its own carrier and network node.

In [ ]:
print_entity(model, "domain.gas.ch")
print_entity(model, "bus.ch.gas")
print_entity(model, "domain.heat.ch")
print_entity(model, "bus.ch.heat")
print_class_counts(model, "After gas and heat domains")

## 11. Add gas supply, CHP conversion, and heat demand

An external gas supply feeds a CHP unit that connects the gas, electricity, and
heat domains. The conversion structure is represented entirely by explicit
attributes and semantic relations.

In [ ]:
gas_bus = model.get_entity("bus.ch.gas")
electricity_bus = model.get_entity("bus.ch")
heat_bus = model.get_entity("bus.ch.heat")

gas_supply = model.add_entity("ExternalSupply", "supply.ch.gas")
gas_supply.name = "Swiss gas import"
gas_supply.supply_capacity = (10_000.0, "MW")
gas_supply.is_slack = True
gas_supply.hasOutputCarrier = Carriers.CARRIER_FUEL_FOSSIL_GAS_NATURAL_GAS
gas_supply.atNode = gas_bus

chp = model.add_entity("CHPUnit", "chp.ch")
chp.name = "Swiss CHP plant"
chp.nominal_electrical_power_capacity = (350.0, "MW")
chp.nominal_thermal_power_capacity = (450.0, "MW")
chp.electrical_efficiency = 0.35
chp.thermal_efficiency = 0.45
chp.total_efficiency = 0.80
chp.power_to_heat_ratio = 350.0 / 450.0
chp.hasInputCarrier = Carriers.CARRIER_FUEL_FOSSIL_GAS_NATURAL_GAS
chp.hasElectricityOutputCarrier = Carriers.CARRIER_ELECTRICITY
chp.hasHeatOutputCarrier = "carrier.heat"
chp.atFuelNode = gas_bus
chp.atElectricityNode = electricity_bus
chp.atHeatNode = heat_bus

heat_demand = model.add_entity("DemandUnit", "dem.ch.heat")
heat_demand.name = "Swiss heat demand"
heat_demand.annual_energy_demand = (20_000_000.0, "MWh/year")
heat_demand.atNode = heat_bus

### Outcome
The CHP unit couples gas, electricity, and heat. Carrier relations describe conversion; node relations describe physical connection points.

In [ ]:
print_entity(model, "supply.ch.gas")
print_entity(model, "chp.ch")
print_entity(model, "dem.ch.heat")
print_class_counts(model, "After CHP and heat demand")

## 12. Validate and export

After construction, the complete model is validated against the CESDM schemas and
exported as hierarchical YAML and a Frictionless Data Package.

In [ ]:
errors = model.validate()
if errors:
    print(f"{len(errors)} validation issue(s):")
    for error in errors[:20]:
        print(" -", error)
    raise SystemExit(1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.export_yaml_hierarchical(
    OUTPUT_DIR / "ch_neighbours_2030.yaml",
)
model.export_frictionless(
    OUTPUT_DIR / "frictionless",
    name="ch-neighbours-2030",
    title="CH + Neighbours 2030 — CESDM tutorial model",
)

print("Model validated and exported successfully.")
print(model.summary())

### Validation and export outcome
Successful validation means all entities, attributes, and relations conform to the loaded schemas. The following cell confirms the generated artifacts.

In [ ]:
print("Export directory:", OUTPUT_DIR)
yaml_file = OUTPUT_DIR / "ch_neighbours_2030.yaml"
frictionless_file = OUTPUT_DIR / "frictionless" / "datapackage.json"
print("YAML exists:", yaml_file.exists(), yaml_file)
print("Frictionless package exists:", frictionless_file.exists(), frictionless_file)

## 13. Inspect the completed model

The model can now be explored using the standard summary and lookup functions.

In [ ]:
print(model.summary())
print()
print(model.summary(detailed=True))

## What this notebook demonstrates

Every part of the model—regions, Carrier Domains, buses, demand, generation,
storage, profiles, interconnectors, and conversion assets—is constructed using:

```python
model.add_entity(...)
model.add_attribute(...)
model.add_relation(...)
```

Convenience functions and the Proxy API can make application code shorter, but
they operate on the same underlying EAR representation and are not required to
construct a complete CESDM model.


## Final model walkthrough

The final cells summarize the completed model and inspect representative entities across several domains.

In [ ]:
print_class_counts(model, "Completed CESDM model")

In [ ]:
for entity_id in [
    "region.ch",
    "domain.electricity",
    "bus.ch",
    "gen.ch.gas",
    "storage.ch.hydro.reservoir",
    "chp.ch",
]:
    print_entity(model, entity_id)

## Key lesson

Despite spanning geography, electricity, gas, heat, demand, generation, storage, profiles, and conversion, the model was built with the same three operations throughout:

```python
model.add_entity(...)
model.add_attribute(...)
model.add_relation(...)
```

The domain meaning comes from the schema-defined classes, attributes, relations, and constraints.
